# SIGMOD Exp 1: Design Space

Microbenchmark over maintained derived-state operations. This notebook wraps `htap_wkld` and compares `SNAP`, `IVMH`, `MONO`, `DUAL`, and `EPOCH` across three workload mixes: read-heavy, balanced, and write-heavy.

The figure is a 3-panel grouped stacked bar chart using the repo's existing paper style.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_REPEAT,
    SIGMOD_TRIM,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    display_name,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp1_design_space').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASELINES = {'naive', 'ivmh'}

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'probe_ratio': 0.001,
    'scan_reuse_ratio': 0.5,
    'analytical_uniform': True,
    'txn_gc_ratio': 0.05,
    'repeat': SIGMOD_REPEAT,
    'trim': SIGMOD_TRIM,
    'readable_every': SIGMOD_READABLE_EVERY,
    'force_rerun': False,
}

WORKLOADS = {
    'RH': 0.80,
    'B': 0.50,
    'WH': 0.20,
}
WORKLOAD_LABELS = {
    'RH': 'Read-Heavy',
    'B': 'Balanced',
    'WH': 'Write-Heavy',
}

NORMALIZATION_TAG = 'aggptx'
REPEAT = CONFIG['repeat']
BASE_ARGS = [
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--scan-reuse-ratio', str(CONFIG['scan_reuse_ratio']),
    '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]
if CONFIG['analytical_uniform']:
    BASE_ARGS += ['--analytical-uniform', 'on']

TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}
READ_TX_ORDER = ['RecentScan', 'HistoryScan', 'Probe', 'DeltaScan']
BUILD_REASON_TARGET = {
    'Probe': 'Probe',
    'HistoryScan': 'HistoryScan',
    'DeltaScan': 'DeltaScan',
    'RecentScan': 'RecentScan',
    'Scan': 'RecentScan',
}
PLOT_TX_ORDER = [
    'InitLoad',
    'Update',
    'GbgCollect',
    'ResidualBuildSnap',
    'RecentScan_build',
    'RecentScan_exec',
    'HistoryScan_build',
    'HistoryScan_exec',
    'Probe_build',
    'Probe_exec',
    'DeltaScan_build',
    'DeltaScan_exec',
]
COLOR_MAP = {
    'InitLoad': TOL['grey'],
    'Update': TOL['yellow'],
    'GbgCollect': TOL['cyan'],
    'ResidualBuildSnap': TOL['purple'],
    'RecentScan_build': TOL['green'],
    'RecentScan_exec': TOL['green'],
    'HistoryScan_build': TOL['darkgreen'],
    'HistoryScan_exec': TOL['darkgreen'],
    'Probe_build': TOL['red'],
    'Probe_exec': TOL['red'],
    'DeltaScan_build': TOL['blue'],
    'DeltaScan_exec': TOL['blue'],
}
HATCH_MAP = {
    'RecentScan_exec': '///',
    'HistoryScan_exec': '\\',
    'Probe_exec': 'xxx',
    'DeltaScan_exec': '\\',
}
LABEL_MAP = {
    'InitLoad': 'InitLoad',
    'Update': 'Update',
    'GbgCollect': 'GC',
    'ResidualBuildSnap': 'Build (Other)',
    'RecentScan_build': 'RecentBuild',
    'RecentScan_exec': 'RecentScan',
    'HistoryScan_build': 'HistoryBuild',
    'HistoryScan_exec': 'HistoryScan',
    'Probe_build': 'JoinBuild',
    'Probe_exec': 'JoinProbe',
    'DeltaScan_build': 'DeltaBuild',
    'DeltaScan_exec': 'DeltaCompare',
}
WRITE_OPS = {'InitLoad', 'Update', 'GbgCollect', 'ResidualBuildSnap', 'RecentScan_build', 'HistoryScan_build', 'Probe_build', 'DeltaScan_build'}
TABLE_ORDER = ['naive', 'ivmh', 'heap', 'chain', 'par']
REPAIR_ORDER = {
    'naive': [''],
    'ivmh': [''],
    'heap': ['No Repair', 'Read Repair', 'Write Repair'],
    'chain': ['Write Repair'],
    'par': ['No Repair', 'Read Repair', 'Write Repair'],
}
REPAIR_LABEL = {'': '', 'No Repair': 'NR', 'Read Repair': 'RR', 'Write Repair': 'WR'}


def token(value):
    if isinstance(value, float):
        return f'{value:g}'.replace('.', 'p')
    return str(value)


RUN_STAMP = current_run_stamp()

RUN_TAG = '_'.join([
    f"wc{token(CONFIG['warehouse_count'])}",
    f"tc{token(CONFIG['txn_count'])}",
    f"bn{token(CONFIG['bucket_num'])}",
    f"ur{token(CONFIG['update_ratio'])}",
    f"pr{token(CONFIG['probe_ratio'])}",
    f"sr{token(CONFIG['scan_reuse_ratio'])}",
    f"au{int(CONFIG['analytical_uniform'])}",
    f"gc{token(CONFIG['txn_gc_ratio'])}",
    f"re{token(CONFIG['readable_every'])}",
    f"rep{token(CONFIG['repeat'])}",
    NORMALIZATION_TAG,
    RUN_STAMP,
])

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('STAMP  :', RUN_STAMP)
print('TAG    :', RUN_TAG)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')

In [ ]:
def collapse_repairs(df, table_type):
    if table_type in BASELINES:
        collapsed = df.groupby(['table_type', 'tx_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'tx_type', 'duration_ms']]
    return df


def collapse_buildsnap_repairs(df, table_type):
    if df.empty:
        return df
    if table_type in BASELINES:
        collapsed = df.groupby(['table_type', 'build_reason'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'build_reason', 'duration_ms']]
    return df


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def aggregate_per_tx(df):
    tx_counts = (
        df.groupby(['table_type', 'repair_type'], as_index=False)
        .size()
        .rename(columns={'size': 'total_tx_count'})
    )
    by_type = (
        df.groupby(['table_type', 'repair_type', 'tx_type'], as_index=False)['duration_ms']
        .sum()
        .merge(tx_counts, on=['table_type', 'repair_type'], how='left')
    )
    by_type['duration_ms'] = by_type['duration_ms'] / by_type['total_tx_count']
    return by_type[['table_type', 'repair_type', 'tx_type', 'duration_ms']]


def aggregate_buildsnap_by_reason(df):
    builds = df[df['tx_type'] == 'BuildSnap'].copy()
    if builds.empty:
        return pd.DataFrame(columns=['table_type', 'repair_type', 'build_reason', 'duration_ms'])
    builds['build_reason'] = builds['build_reason'].fillna('Unknown')
    tx_counts = (
        df.groupby(['table_type', 'repair_type'], as_index=False)
        .size()
        .rename(columns={'size': 'total_tx_count'})
    )
    by_reason = (
        builds.groupby(['table_type', 'repair_type', 'build_reason'], as_index=False)['duration_ms']
        .sum()
        .merge(tx_counts, on=['table_type', 'repair_type'], how='left')
    )
    by_reason['duration_ms'] = by_reason['duration_ms'] / by_reason['total_tx_count']
    return by_reason[['table_type', 'repair_type', 'build_reason', 'duration_ms']]



def load_buildsnap_breakdown(name):
    path = DATA_DIR / f'sigmod_exp1_buildsnap_{name}_{RUN_TAG}.csv'
    if path.exists():
        return pd.read_csv(path, keep_default_na=False)
    print(f'Warning: missing BuildSnap breakdown CSV for {name}; rerun with force_rerun=True to regenerate it.')
    return pd.DataFrame(columns=['table_type', 'repair_type', 'build_reason', 'duration_ms'])


def build_plot_df(df, builds_df):
    build_map = {}
    if not builds_df.empty:
        build_rows = builds_df.copy()
        build_rows['target_tx_type'] = build_rows['build_reason'].map(BUILD_REASON_TARGET)
        build_rows = build_rows[build_rows['target_tx_type'].notna()].copy()
        build_rows = (
            build_rows.groupby(['table_type', 'repair_type', 'target_tx_type'], as_index=False)['duration_ms']
            .sum()
        )
        build_map = {
            (row.table_type, row.repair_type, row.target_tx_type): float(row.duration_ms)
            for row in build_rows.itertuples(index=False)
        }

    rows = []
    attributed_build = {}
    total_build = {}
    build_rows_df = df[df['tx_type'] == 'BuildSnap']
    if not build_rows_df.empty:
        total_build = {
            (row.table_type, row.repair_type): float(row.duration_ms)
            for row in build_rows_df.itertuples(index=False)
        }

    for row in df.itertuples(index=False):
        tx = row.tx_type
        base = {
            'table_type': row.table_type,
            'repair_type': row.repair_type,
        }
        if tx == 'BuildSnap':
            continue
        if tx in READ_TX_ORDER:
            build_value = float(build_map.get((row.table_type, row.repair_type, tx), 0.0))
            build_value = min(build_value, float(row.duration_ms))
            exec_value = max(float(row.duration_ms) - build_value, 0.0)
            if build_value > 0:
                rows.append({**base, 'tx_type': f'{tx}_build', 'duration_ms': build_value})
            rows.append({**base, 'tx_type': f'{tx}_exec', 'duration_ms': exec_value})
            attributed_build[(row.table_type, row.repair_type)] = attributed_build.get((row.table_type, row.repair_type), 0.0) + build_value
            continue
        rows.append({**base, 'tx_type': tx, 'duration_ms': float(row.duration_ms)})

    for key, total in total_build.items():
        residual = max(total - attributed_build.get(key, 0.0), 0.0)
        if residual > 1e-9:
            rows.append({
                'table_type': key[0],
                'repair_type': key[1],
                'tx_type': 'ResidualBuildSnap',
                'duration_ms': residual,
            })

    return pd.DataFrame(rows, columns=['table_type', 'repair_type', 'tx_type', 'duration_ms'])


def run_workload(name, analytical_ratio):
    csv_path = DATA_DIR / f'sigmod_exp1_{name}_{RUN_TAG}.csv'
    buildsnap_csv_path = DATA_DIR / f'sigmod_exp1_buildsnap_{name}_{RUN_TAG}.csv'
    if CONFIG['force_rerun'] and csv_path.exists():
        csv_path.unlink()
    if CONFIG['force_rerun'] and buildsnap_csv_path.exists():
        buildsnap_csv_path.unlink()
    if csv_path.exists():
        print(f'Using cached CSV: {csv_path.name}')
        if not buildsnap_csv_path.exists():
            print(f'Warning: missing BuildSnap breakdown CSV for {name}; rerun with force_rerun=True to regenerate it.')
        return pd.read_csv(csv_path, keep_default_na=False)

    args = BASE_ARGS + ['--analytical-ratio', str(analytical_ratio)]
    rows = []
    build_rows = []
    for table_type in TABLE_TYPES:
        print(f'  workload={name} table={table_type}')
        trials = []
        raw_trials = []
        for trial in range(REPEAT):
            result = run_checked([str(BIN), *args, '--table-type', table_type], ROOT, quiet=True)
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {name}/{table_type}')
            df['tx_type'] = df['tx_type'].replace(TX_MAP)
            df['trial'] = trial
            raw_trials.append(df)
            agg = aggregate_per_tx(df)
            agg['trial'] = trial
            trials.append(agg)
        df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
        df_raw = trim_trial_runs(pd.concat(raw_trials, ignore_index=True))
        df_avg = (
            df_all.groupby(['table_type', 'repair_type', 'tx_type'], as_index=False)['duration_ms']
            .mean()
        )
        rows.append(collapse_repairs(df_avg, table_type))
        build_df = aggregate_buildsnap_by_reason(df_raw)
        if not build_df.empty:
            build_rows.append(collapse_buildsnap_repairs(build_df, table_type))

    out = pd.concat(rows, ignore_index=True)
    out.to_csv(csv_path, index=False)
    print(f'Saved {csv_path.name}')
    if build_rows:
        buildsnap_out = pd.concat(build_rows, ignore_index=True)
        buildsnap_out.to_csv(buildsnap_csv_path, index=False)
        print(f'Saved {buildsnap_csv_path.name}')
    return out


results = {name: run_workload(name, ratio) for name, ratio in WORKLOADS.items()}
plot_results = {
    name: build_plot_df(results[name], load_buildsnap_breakdown(name))
    for name in WORKLOADS
}
summary = []
for name, df in results.items():
    total = df.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].sum()
    total['workload'] = name
    summary.append(total)
display(pd.concat(summary, ignore_index=True))


In [ ]:
def build_legend_handles(df):
    present = set(df['tx_type'].unique())
    legend_handles = []
    legend_labels = []
    for tx in PLOT_TX_ORDER[::-1]:
        if tx not in present:
            continue
        if tx in WRITE_OPS:
            patch = Patch(facecolor=COLOR_MAP[tx], edgecolor='black', linewidth=0.4)
        else:
            patch = Patch(facecolor='white', edgecolor=COLOR_MAP[tx], linewidth=0.9, hatch=HATCH_MAP.get(tx, '///'))
        legend_handles.append(patch)
        legend_labels.append(LABEL_MAP[tx])
    return legend_handles, legend_labels


def panel_total_max(df):
    pivot = df.pivot_table(index=['table_type', 'repair_type'], columns='tx_type', values='duration_ms', aggfunc='sum', fill_value=0.0)
    pivot = pivot.reindex(columns=PLOT_TX_ORDER, fill_value=0.0)
    if pivot.empty:
        return 0.0
    return float(pivot.sum(axis=1).max())


def plot_panel(ax, df, y_max):
    pivot = df.pivot_table(index=['table_type', 'repair_type'], columns='tx_type', values='duration_ms', aggfunc='sum', fill_value=0.0)
    pivot = pivot.reindex(columns=PLOT_TX_ORDER, fill_value=0.0)

    bar_x, minor_labels, major_centers, major_labels, combos = [], [], [], [], []
    x = 0.0
    gap = 0.36
    for table in TABLE_ORDER:
        subs = REPAIR_ORDER[table]
        start = x
        for repair in subs:
            combos.append((table, repair))
            bar_x.append(x)
            minor_labels.append(REPAIR_LABEL[repair])
            x += 0.78
        major_centers.append((start + (x - 0.78)) / 2.0)
        major_labels.append(display_name(table, ''))
        x += gap

    for idx, (table, repair) in enumerate(combos):
        if (table, repair) in pivot.index:
            row = pivot.loc[(table, repair)]
        else:
            row = pd.Series(0.0, index=PLOT_TX_ORDER)
        bottom = 0.0
        for tx in PLOT_TX_ORDER:
            value = float(row.get(tx, 0.0))
            if value <= 0:
                continue
            if tx in WRITE_OPS:
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color=COLOR_MAP[tx], edgecolor='black', linewidth=0.4)
            else:
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='white', edgecolor='black', linewidth=0.4)
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='none', edgecolor=COLOR_MAP[tx], linewidth=0.9, hatch=HATCH_MAP.get(tx, '///'))
            bottom += value

    ax.set_xticks([])
    ax.set_ylabel('Duration (ms / tx)')
    ax.set_ylim(0, y_max)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    for xi, label in zip(bar_x, minor_labels):
        ax.text(xi, -0.06, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=8, clip_on=False)
    for xc, label in zip(major_centers, major_labels):
        ax.text(xc, -0.14, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=9, clip_on=False)


global_y_max = max(panel_total_max(plot_results[name]) for name in WORKLOADS) * 1.08

for name in WORKLOADS:
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.4))
    plot_df = plot_results[name]
    plot_panel(ax, plot_df, global_y_max)
    legend_handles, legend_labels = build_legend_handles(plot_df)
    ax.legend(legend_handles, legend_labels, loc='upper right', ncol=2, fontsize=7, framealpha=0.95)
    fig.tight_layout(rect=[0, 0, 1, 1])
    out_pdf = FIGS_DIR / f'sigmod_exp1_design_space_{name.lower()}_{RUN_TAG}.pdf'
    fig.savefig(out_pdf, format='pdf', bbox_inches='tight')
    plt.show()
    print('Saved', out_pdf)


In [ ]:
SNAPSTAT_TABLES = ['naive', 'ivmh']


def run_snapshot_stats(name, analytical_ratio):
    csv_path = DATA_DIR / f'sigmod_exp1_snapshot_stats_{name}_{RUN_TAG}.csv'
    if CONFIG['force_rerun'] and csv_path.exists():
        csv_path.unlink()
    if csv_path.exists():
        print(f'Using cached snapshot stats: {csv_path.name}')
        return pd.read_csv(csv_path, keep_default_na=False)

    args = BASE_ARGS + ['--analytical-ratio', str(analytical_ratio)]
    rows = []
    for table_type in SNAPSTAT_TABLES:
        tmp_path = DATA_DIR / f'sigmod_exp1_snapshot_stats_{name}_{table_type}_{RUN_TAG}.csv'
        if tmp_path.exists():
            tmp_path.unlink()
        print(f'  snapshot-stats workload={name} table={table_type}')
        run_checked([
            str(BIN),
            *args,
            '--table-type', table_type,
            '--snapshot-stat', str(tmp_path),
        ], ROOT, quiet=True)
        df = pd.read_csv(tmp_path, keep_default_na=False)
        df['workload'] = name
        rows.append(df)

    out = pd.concat(rows, ignore_index=True)
    out.to_csv(csv_path, index=False)
    print(f'Saved {csv_path.name}')
    return out


snapshot_stats = {
    name: run_snapshot_stats(name, ratio)
    for name, ratio in WORKLOADS.items()
}
snapshot_df = pd.concat(snapshot_stats.values(), ignore_index=True)
snapshot_df['workload'] = snapshot_df['workload'].map(WORKLOAD_LABELS)
snapshot_df['table_label'] = snapshot_df['table_type'].map({'naive': 'SNAP', 'ivmh': 'IVMH'})
snapshot_df['avg_reads_per_retained_snapshot'] = (
    snapshot_df['snapshot_reads_total']
    / snapshot_df['retained_snapshots'].replace(0, np.nan)
)
snapshot_df['avg_reads_per_built_snapshot'] = (
    snapshot_df['snapshot_reads_total']
    / snapshot_df['snapshots_built_total'].replace(0, np.nan)
)

display(
    snapshot_df[[
        'workload',
        'table_label',
        'readable_timestamps_published',
        'retained_snapshots',
        'snapshots_built_total',
        'snapshot_reads_total',
        'snapshot_cache_hits_total',
        'snapshot_cache_misses_total',
        'current_reads_total',
        'avg_reads_per_retained_snapshot',
        'avg_reads_per_built_snapshot',
    ]]
    .sort_values(['workload', 'table_label'])
    .reset_index(drop=True)
)

buildsnap_frames = []
for name in WORKLOADS:
    path = DATA_DIR / f'sigmod_exp1_buildsnap_{name}_{RUN_TAG}.csv'
    if path.exists():
        df = pd.read_csv(path, keep_default_na=False)
        df['workload'] = name
        buildsnap_frames.append(df)

if buildsnap_frames:
    buildsnap_df = pd.concat(buildsnap_frames, ignore_index=True)
    buildsnap_df['target_tx_type'] = buildsnap_df['build_reason'].map(BUILD_REASON_TARGET)
    result_lookup = pd.concat([
        df.assign(workload=name)
        for name, df in results.items()
    ], ignore_index=True)
    buildsnap_df = buildsnap_df.merge(
        result_lookup[['workload', 'table_type', 'repair_type', 'tx_type', 'duration_ms']].rename(
            columns={'tx_type': 'target_tx_type', 'duration_ms': 'target_tx_ms'}
        ),
        on=['workload', 'table_type', 'repair_type', 'target_tx_type'],
        how='left',
    )
    buildsnap_df['build_vs_target_pct'] = (
        100.0 * buildsnap_df['duration_ms']
        / buildsnap_df['target_tx_ms'].replace(0, np.nan)
    )
    buildsnap_df['workload'] = buildsnap_df['workload'].map(WORKLOAD_LABELS)
    buildsnap_df['table_label'] = buildsnap_df['table_type'].map({'naive': 'SNAP', 'ivmh': 'IVMH'})
    display(
        buildsnap_df[[
            'workload',
            'table_label',
            'repair_type',
            'build_reason',
            'duration_ms',
            'target_tx_ms',
            'build_vs_target_pct',
        ]]
        .sort_values(['workload', 'table_label', 'repair_type', 'build_reason'])
        .reset_index(drop=True)
    )
else:
    print('No BuildSnap breakdown CSVs found for this run.')
